# **Diplomado IA: Aplicaciones 2 - Audio y Video**. <br> Práctico 44: Deep Fakes - Clonación de voz
---
---

**Profesor:**
- Carlos Aspillaga


Nota: Algunas partes de este laboratorio se construyeron usando recursos de las siguientes fuentes:
* https://colab.research.google.com/github/tugstugi/dl-colab-notebooks/blob/master/notebooks/RealTimeVoiceCloning.ipynb
* https://towardsdatascience.com/how-to-produce-a-deepfake-video-in-5-minutes-513984fd24b6
* https://colab.research.google.com/github/Tyler-Hilbert/AudioProcessingInPythonWorkshop/blob/master/AudioProcessingInPython.ipynb#scrollTo=AEvn0yZKNCV4

# **1. Clonación de Voz**

Para clonar la voz de Obama, usaremos un modelo preentrenado

❗❗❗**Nota importante**: ❗❗❗

Ir a "Entorno de Ejecución" > "Cambiar entorno de ejecución"  y seleccionar la versión 2025.07 de Google Colab

### Descarga de código e instalación de librerías necesarias

In [1]:
!pip3 install -U scipy
!git clone https://github.com/jnordberg/tortoise-tts.git
%cd tortoise-tts
!pip install numba
!pip install llvmlite
!pip install transformers==4.29.2
!pip3 install -r requirements.txt
!pip3 install einops==0.5.0
!pip3 install rotary_embedding_torch==0.1.5 unidecode==1.3.5
!python3 setup.py install
!pip install audio2numpy
!apt-get install -qq libportaudio2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 23.2 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
plotnine 0.14.6 requires scipy<1.16.0,>=1.8.0, but you have scipy 1.17.1 which is incompatible.
Cloning into 'tortoise-tts'...
remote: Enumerating objects: 1481, done.
remote: Total 1481 (delta 0), reused 0 (delta 0), pack-reused 1481 (from 1)
Receiving objects: 100% (1481/1481), 53.56 MiB | 28.76 MiB/s, done.
Resolving deltas: 100% (604/604), done.
Updating files: 100% (558/558), done.
/content/tortoise-tts
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install "tokenizers==0.13.3" "transformers==4.29.2"

DEPRECATION: Loading egg at /usr/local/lib/python3.11/dist-packages/progressbar-2.5-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.11/dist-packages/TorToiSe-2.3.0-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [3]:
from transformers import __version__
print(__version__)

4.29.2


In [4]:
import os
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
import IPython
from tortoise.api import TextToSpeech
from tortoise.utils.audio import load_audio, load_voice, load_voices
import sys
from IPython.display import display, Audio, clear_output
from IPython.utils import io
import ipywidgets as widgets
import numpy as np
from pathlib import Path
import soundfile as sf
import audio2numpy as a2n


Como input al modelo preentrenado, necesitamos entregar un audio de referencia, para que el modelo pueda calcular un descriptor de la voz.

Ejecutando las siguientes celdas, usted estará descargando el archivo de Demo provisto por el profesor. Se descargarán los primeros 90 segundos del video https://www.youtube.com/watch?v=sTFWC1PiLVE donde aparece Barack Obama hablando.

## Video de referencia de voz (Demo)

In [5]:
%%html
<iframe width="560" height="315" src="https://www.youtube.com/embed/sTFWC1PiLVE?si=ZWuMADnnYIPTkE7h" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

In [6]:
if not os.path.exists("/content/obama_voice.mp3"):
    os.system('wget -q https://www.dropbox.com/s/bxa9ivwuveu3s3q/obama_voice.mp3 -O /content/obama_voice.mp3')
    os.system('ffmpeg -y -loglevel error -stats -i /content/obama_voice.mp3 -ac 1 -ab 64000 -ar 22050 -t 90 /content/reference_voice.wav')
    audio, sampling_rate = a2n.audio_from_file("/content/reference_voice.wav")
    display(Audio(audio, rate=22050, autoplay=False))

Output hidden; open in https://colab.research.google.com to view.

A continuación particionamos el audio en 3 clips de 10 segundos cada uno

In [7]:
!mkdir /content/tortoise-tts/tortoise/voices/obama
!ffmpeg -loglevel error -y -ss 0 -t 10 -i /content/reference_voice.wav /content/tortoise-tts/tortoise/voices/obama/clip_0.wav
!ffmpeg -loglevel error -y -ss 40 -t 10 -i /content/reference_voice.wav /content/tortoise-tts/tortoise/voices/obama/clip_1.wav
!ffmpeg -loglevel error -y -ss 80 -t 10 -i /content/reference_voice.wav /content/tortoise-tts/tortoise/voices/obama/clip_2.wav

A continuación instanciamos el modelo (y se descargan los pesos preentrenados)

In [8]:
tts = TextToSpeech()
voice_samples, conditioning_latents = load_voice('obama')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

...

Done.


100% (25193729 of 25193729) |############| Elapsed Time: 0:00:00 Time:  0:00:00


Done.


100% (100715777 of 100715777) |##########| Elapsed Time: 0:00:01 Time:  0:00:01


Done.


---
## Actividad: Texto personalizado para la generación de voz

Para cumplir con los requisitos de la actividad, se elige una **frase diferente** a la usada en clases.

| Parámetro | Valor elegido |
|---|---|
| **Texto a sintetizar** | *"Se hace más fácil. Cada día se hace un poco más fácil. Pero hay que hacerlo todos los días; esa es la parte difícil. Pero sí, se hace más fácil"* |
| **Voz de referencia** | Barack Obama (discurso de YouTube — permitido según enunciado) |
| **Preset de calidad** | `ultra_fast` (suficiente para demo; usar `fast` para mejor calidad) |

### ¿Cómo funciona Tortoise TTS?

**Tortoise TTS** es un modelo de síntesis de voz con clonación (*voice cloning*) de pocos ejemplos (*few-shot*):

```
Audio de referencia (3 clips × 10s)
        ↓
   Encoder de voz  ──► conditioning_latents  (descriptor de la voz)
        ↓
   Tortoise TTS  ◄── texto de entrada
        ↓
   Audio generado con la voz clonada
```

El modelo **no necesita reentrenarse**: solo usa los clips de referencia para extraer un embedding
de la voz y condicionar la generación. Esto es posible gracias a un entrenamiento previo con miles
de voces distintas que lo volvieron generalizable.

> 💡 **Tip**: Si el audio suena robótico, ejecute la celda nuevamente. Tortoise es estocástico
> y cada ejecución produce un resultado diferente.

---

A continuación podemos entregar un texto de referencia y el modelo nos generará la voz (en este caso de Obama) diciendo esa frase.

*Nota: Cada vez que se ejecuta el código se genera un audio diferente.*

In [13]:
text = "It gets easier. Every day it gets a little easier. But you gotta do it every day—that’s the hard part. But yeah, it does get easier."
preset = "ultra_fast" # Options: {"ultra_fast", "fast" (default), "standard", "high_quality"}
gen = tts.tts_with_preset(text, voice_samples=voice_samples, conditioning_latents=conditioning_latents, preset=preset)
torchaudio.save(f'/content/generated.wav', gen.squeeze(0).cpu(), 24000)
IPython.display.Audio(f'/content/generated.wav')

Generating autoregressive samples..


100%|██████████| 1/1 [05:52<00:00, 352.47s/it]


Computing best candidates using CLVP and CVVP


100%|██████████| 1/1 [00:01<00:00,  1.42s/it]


Transforming autoregressive outputs into audio..


100%|██████████| 30/30 [00:08<00:00,  3.44it/s]
